## Data preparation (GDAL)

In [ ]:
 gdalwarp -te -180 -90 180 90 -te_srs 'EPSG:4326' -t_srs 'ESRI:54008' -tr 30 30 /vsicurl/http://192.168.49.35:8333/global/lc/land.cover_ifad.mcdt_c_30m_s_20200101_20201231_go_epsg.4326_v1.1.tif land.cover_ifad.mcdt_c_30m_s_20200101_20201231_go_esri.54008_v1.1.vrt

gdalwarp -te -180 -90 180 90 -te_srs 'EPSG:4326' -t_srs 'ESRI:54008' -tr 30 30 /vsicurl/http://192.168.49.34:8333/gpw/arco/gpw_short.veg.height_egbt.trend.mk_m_30m_s_20000101_20221231_go_epsg.4326_v20240809.tif gpw_short.veg.height_egbt.trend.mk_m_30m_s_20
000101_20221231_go_esri.54008_v20240809.vrt 

gdalwarp -te -180 -90 180 90 -te_srs 'EPSG:4326' -t_srs 'ESRI:54008' -tr 30 30 /vsicurl/http://192.168.49.36:8333/global/continent.mask/continent.mask_c_30m_s_20200101_20201231_go_epsg.4326_v20260629.tif continent.mask_c_30m_s_20200101_20201231_go_esri.54008_v20260629.vrt

## Tile system

In [1]:
import rasterio 
from skmap.parallel.utils import TilingProcessing

/skmap/lib/python3.14/site-packages/skmap/misc.py:209: UserWarning: ERROR: No module named 'pytz'

Encountered because misc.GoogleSheet has additional dependencies, please try:

	pip install skmap[full]

  warnings.warn(


In [2]:
base_raster = 'continent.mask_c_30m_s_20200101_20201231_go_esri.54008_v20260629.vrt'
src = rasterio.open(base_raster)

In [ ]:
tiles = TilingProcessing.generate_tiles(tile_size=(10000*30), extent = src.bounds, crs=src.crs, raster_layer_fn=base_raster)

/skmap/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/skmap/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/skmap/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/skmap/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/skmap/lib/python3.14/si

In [7]:
tiles.to_file('tiles_1km_esri.54008.gpkg')

## Preparation of arguments

In [58]:
import rasterio 
import numpy as np
from rasterio.windows import from_bounds

import geopandas as gpd
tiles = gpd.read_file('tiles_1km_esri.54008.gpkg')
base_raster = 'continent.mask_c_30m_s_20200101_20201231_go_esri.54008_v20260629.vrt'

args = []
for _,t in tiles.iterrows():
    bounds = t.geometry.bounds
    src = rasterio.open(base_raster)
    window = from_bounds(bounds[0], bounds[1], bounds[2], bounds[3], src.transform).round_shape().round_offsets()
    args.append([t.tile_id, window.col_off, window.row_off, window.width, window.height])

np.savetxt("args.in", np.array(args), fmt="%s")

/skmap/lib/python3.14/site-packages/rasterio/windows.py:729: RasterioDeprecationWarning: round_shape is deprecated and will be removed in Rasterio 2.0.0.
  warnings.warn(


## Pixel counting

In [ ]:
#!uv pip install --python /skmap/bin/python pyarrow

In [11]:
import numpy as np
arg_fn = '/mnt/atlas/WRI/args.in'
args = np.loadtxt(arg_fn, dtype=str)

In [ ]:
from rasterio.windows import Window
from skmap.io import read_rasters_cpp
from skmap.misc import ttprint
from numba import njit
import numpy as np
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import Table

@njit(parallel = True, fastmath = True)
def unique_counts_bitwise(arr):
  
  packed = (arr[0].astype(np.int32) << 16) | (arr[1].astype(np.int32) << 8) | arr[2].astype(np.int32)
  sorted_packed = np.sort(packed)

  n = len(sorted_packed)
  uniques = [sorted_packed[0]]
  counts = [1]
  for i in range(1, n):
    if sorted_packed[i] == sorted_packed[i - 1]:
      counts[-1] += 1
    else:
      uniques.append(sorted_packed[i])
      counts.append(1)

  packed_arr = np.array(uniques)
  return packed_arr >> 16 & 0xFF, packed_arr >> 8 & 0xFF, packed_arr & 0xFF, np.array(counts)

out_file = './gpw_sampling.design.pq'

raster_files = [
    'land.cover_ifad.mcdt_c_30m_s_20200101_20201231_go_esri.54008_v1.1.vrt',
    'continent.mask_c_30m_s_20200101_20201231_go_esri.54008_v20260629.vrt',
    'gpw_short.veg.height_egbt.trend.mk_m_30m_s_20000101_20221231_go_esri.54008_v20240809.vrt'
]

gdal_opts = {
    'CPL_VSIL_CURL_ALLOWED_EXTENSIONS': '.tif',
    'GDAL_DISABLE_READDIR_ON_OPEN': 'EMPTY_DIR',
    'CPL_VSIL_CURL_USE_HEAD': 'NO'
}

i0 = 18650
i1 = i0+10

ttprint("Start")

for arg in args[i0:i1]:
    tile, col_off, row_off = arg[0], int(arg[1]), int(arg[2])
    ttprint(tile, col_off, row_off)
    window = Window(col_off=col_off, row_off=row_off, width=4000, height=4000)
    #print(window)

    data = read_rasters_cpp(raster_files=raster_files, window=window, gdal_opts=gdal_opts, verbose=False)
    data[0,:] = np.where(data[0] > 0, 2, np.where(data[0] < 0, 1, 0))
    data[1,:] = np.where(np.isnan(data[1]), 0, data[1])
    data[2,:] = np.where(np.isnan(data[2]), 0, data[2])
    
    #print('short_veg: ', np.unique(data[0]))
    #print('land.cover: ', np.unique(data[1]))
    #print('continent.mask: ', np.unique(data[2]))

    d0, d1, d2, counts = unique_counts_bitwise(data)
    
    uniqs  = { Path(r).stem:d for r,d in zip(raster_files,[d0, d1, d2]) }
    uniqs['count'] = counts
    uniqs['tile'] = tile

    uniqs = pd.DataFrame(uniqs)

    pq.write_to_dataset(
      Table.from_pandas(uniqs),
      out_file,
      partition_cols=['tile'],
      compression="snappy",
      version="2.4",
    )

ttprint("End")

## Read all tiles

In [63]:
import pyarrow.dataset as ds
dataset = ds.dataset("gpw_strata.count_esri.54008.pq", format="parquet", partitioning="hive")
table = dataset.to_table()
df_strata = table.to_pandas()

In [65]:
na_mask = np.logical_not(np.logical_or(df_strata.iloc[:,1] == 0, df_strata.iloc[:,2] == 0))
df_strata = df_strata[na_mask]
df_strata

,gpw_short.veg.height_egbt.trend.mk_m_30m_s_20000101_20221231_go_esri.54008_v20240809,land.cover_ifad.mcdt_c_30m_s_20200101_20201231_go_esri.54008_v1.1,continent.mask_c_30m_s_20200101_20201231_go_esri.54008_v20260629,count,tile
5,0,1,5,1755,1021
7,0,10,5,2083,1021
9,0,11,5,165284,1021
11,0,12,5,119,1021
13,0,13,5,857097,1021
...,...,...,...,...,...
233971,2,34,5,130,957
233973,2,35,5,30,957
233975,2,38,5,319,957
233977,2,39,5,119,957


In [66]:
df_strata['gpw_short.veg.height_egbt.trend.mk'] = df_strata['gpw_short.veg.height_egbt.trend.mk_m_30m_s_20000101_20221231_go_esri.54008_v20240809'].map({
    0: 'stable',
    1: 'negative',
    2: 'positive'
})

df_strata['continent.mask'] = df_strata['continent.mask_c_30m_s_20200101_20201231_go_esri.54008_v20260629'].map({
  1: 'Africa',
  2: 'Asia',
  3: 'Australia',
  4: 'North America',
  5: 'Oceania',
  6: 'South America',
  7: 'Antarctica',
  8: 'Europe'
})

df_strata['land.cover_ifad.mcdt_lv2'] = df_strata['land.cover_ifad.mcdt_c_30m_s_20200101_20201231_go_esri.54008_v1.1'].map({
  1: "1.1 Built-up [1]",
  2: "2.1 Annual crops - soybean [2]",
  3: "2.2 Annual crops - maize [3]",
  4: "2.3 Annual crops - rice [4]",
  5: "2.4 Annual crops - other [5]",
  6: "2.5 Perenial crops - coffee [6]",
  7: "2.6 Perenial crops - cocoa [7]",
  8: "2.7 Perenial crops - rubber [8]",
  9: "2.8 Perenial crops - palm [9]",
  10: "3.1 Cultivated grassland [10]",
  11: "3.2 Natural grassland [11]",
  12: "4.1 Tree cover broadleaved evergreen open [12]",
  13: "4.2 Tree cover broadleaved evergreen closed [13]",
  14: "4.3 Tree cover broadleaved deciduous closed to open (>15%) [14]",
  15: "4.4 Tree cover broadleaved deciduous closed (>40%) [15]",
  16: "4.5 Tree cover needleleaved evergreen closed to open (>15%) [16]",
  17: "4.6 Tree cover needleleaved evergreen closed (>40%) [17]",
  18: "4.7 Tree cover needleleaved deciduous closed to open (>15%) [18]",
  19: "4.8 Tree cover needleleaved deciduous closed (>40%) [19]",
  20: "4.9 Tree cover mixed leaf type open (broadleaved and needleleaved) [20]",
  21: "4.10 Tree cover mixed leaf type closed (broadleaved and needleleaved) [21]",
  22: "4.11 Flooded tree cover open [22]",
  23: "4.12 Flooded tree cover closed [23]",
  24: "4.13 Other flooded tree cover [24]",
  25: "4.14 Other tree cover [25]",
  26: "5.1 Shrubland and heathland [26]",
  27: "6.1 True desert [27]",
  28: "6.2 Semi-arid [28]",
  29: "6.3 Semi-arid shrubland [29]",
  30: "6.4 Rocks and mountains peaks [30]",
  31: "6.5 Snow and ice [31]",
  32: "7.1 Wetland - Sparse vegetation [32]",
  33: "7.2 Wetland - Grassland [33]",
  34: "7.3 Wetland - Shrubland [34]",
  35: "8.1 Water - Permanent [35]",
  36: "8.2 Water - Seasonal [36]",
  37: "9.1 Mangrove [37]",
  38: "9.2 Coastal wetlands [38]",
  39: "9.3 Beaches and dunes [39]",
  40: "10.1 Aquaculture [40]"
})

df_strata['land.cover_ifad.mcdt_lv1'] = df_strata['land.cover_ifad.mcdt_c_30m_s_20200101_20201231_go_esri.54008_v1.1'].map({
  1: '1. Built-up [1]',
  2: '2. Cropland (Annual and Perennial) [2-9]',
  3: '2. Cropland (Annual and Perennial) [2-9]',
  4: '2. Cropland (Annual and Perennial) [2-9]',
  5: '2. Cropland (Annual and Perennial) [2-9]',
  6: '2. Cropland (Annual and Perennial) [2-9]',
  7: '2. Cropland (Annual and Perennial) [2-9]',
  8: '2. Cropland (Annual and Perennial) [2-9]',
  9: '2. Cropland (Annual and Perennial) [2-9]',
  10: '3. Grassland [10-11]',
  11: '3. Grassland [10-11]',
  12: '4. Forest, forest plantatioins and woodland [12-25]',
  13: '4. Forest, forest plantatioins and woodland [12-25]',
  14: '4. Forest, forest plantatioins and woodland [12-25]',
  15: '4. Forest, forest plantatioins and woodland [12-25]',
  16: '4. Forest, forest plantatioins and woodland [12-25]',
  17: '4. Forest, forest plantatioins and woodland [12-25]',
  18: '4. Forest, forest plantatioins and woodland [12-25]',
  19: '4. Forest, forest plantatioins and woodland [12-25]',
  20: '4. Forest, forest plantatioins and woodland [12-25]',
  21: '4. Forest, forest plantatioins and woodland [12-25]',
  22: '4. Forest, forest plantatioins and woodland [12-25]',
  23: '4. Forest, forest plantatioins and woodland [12-25]',
  24: '4. Forest, forest plantatioins and woodland [12-25]',
  25: '4. Forest, forest plantatioins and woodland [12-25]',
  26: '5. Shrubland and heathland [26]',
  27: '6. Sparsely vegetated ecosystems [27-31]',
  28: '6. Sparsely vegetated ecosystems [27-31]',
  29: '6. Sparsely vegetated ecosystems [27-31]',
  30: '6. Sparsely vegetated ecosystems [27-31]',
  31: '6. Sparsely vegetated ecosystems [27-31]',
  32: '7. Inland wetlands [32-34]',
  33: '7. Inland wetlands [32-34]',
  34: '7. Inland wetlands [32-34]',
  35: '8. Water bodies [35-36]',
  36: '8. Water bodies [35-36]',
  37: '9. Coastal ecosystems [37-39]',
  38: '9. Coastal ecosystems [37-39]',
  39: '9. Coastal ecosystems [37-39]',
  40: '10. Aquaculture [40]'
})

df_strata['gpw_class'] = df_strata['land.cover_ifad.mcdt_c_30m_s_20200101_20201231_go_esri.54008_v1.1'].map({
  1: "Not grassland",
  2: "Not grassland",
  3: "Not grassland",
  4: "Not grassland",
  5: "Not grassland",
  6: "Not grassland",
  7: "Not grassland",
  8: "Not grassland",
  9: "Not grassland",
  10: "Cultivated grassland",
  11: "Natural grassland/Semi-natural",
  12: "Not grassland",
  13: "Not grassland",
  14: "Not grassland",
  15: "Not grassland",
  16: "Not grassland",
  17: "Not grassland",
  18: "Not grassland",
  19: "Not grassland",
  20: "Not grassland",
  21: "Not grassland",
  22: "Not grassland",
  23: "Not grassland",
  24: "Not grassland",
  25: "Not grassland",
  26: "Natural grassland/Semi-natural",
  27: "Not grassland",
  28: "Not grassland",
  29: "Not grassland",
  30: "Not grassland",
  31: "Not grassland",
  32: "Not grassland",
  33: "Natural grassland/Semi-natural",
  34: "Not grassland",
  35: "Not grassland",
  36: "Not grassland",
  37: "Not grassland",
  38: "Not grassland",
  39: "Not grassland",
  40: "Not grassland"
})

In [67]:
cols = list(df_strata.columns[-5:]) + ['count']
df_strata = df_strata[cols].reset_index()

#na_mask = np.logical_or.reduce([df_strata.iloc[:,2].isnull(),df_strata.iloc[:,3].isnull(),df_strata.iloc[:,4].isnull()])
#df_strata = df_strata[np.logical_not(na_mask)]

df_strata

,index,gpw_short.veg.height_egbt.trend.mk,continent.mask,land.cover_ifad.mcdt_lv2,land.cover_ifad.mcdt_lv1,gpw_class,count
0,5,stable,Oceania,1.1 Built-up [1],1. Built-up [1],Not grassland,1755
1,7,stable,Oceania,3.1 Cultivated grassland [10],3. Grassland [10-11],Cultivated grassland,2083
2,9,stable,Oceania,3.2 Natural grassland [11],3. Grassland [10-11],Natural grassland/Semi-natural,165284
3,11,stable,Oceania,4.1 Tree cover broadleaved evergreen open [12],"4. Forest, forest plantatioins and woodland [1...",Not grassland,119
4,13,stable,Oceania,4.2 Tree cover broadleaved evergreen closed [13],"4. Forest, forest plantatioins and woodland [1...",Not grassland,857097
...,...,...,...,...,...,...,...
160164,233971,positive,Oceania,7.3 Wetland - Shrubland [34],7. Inland wetlands [32-34],Not grassland,130
160165,233973,positive,Oceania,8.1 Water - Permanent [35],8. Water bodies [35-36],Not grassland,30
160166,233975,positive,Oceania,9.2 Coastal wetlands [38],9. Coastal ecosystems [37-39],Not grassland,319
160167,233977,positive,Oceania,9.3 Beaches and dunes [39],9. Coastal ecosystems [37-39],Not grassland,119


In [69]:
df_strata_agg = df_strata[cols].groupby(cols[:-1]).sum().reset_index()
df_strata_agg.to_csv('gpw_strata.csv')